# Lesson 28: Class imbalance, resampling, and class weights

Class imbalance means that one clas is much more common than the other

For example, suppose there are:

$$ 100~fraudulent~transactions $$

and:

$$ 9900~legitimate~transactions $$

Fraud prevalence is:

$$ \frac{100}{10000} = 0.01 = 1\% $$

The dataset is strongly imbalanced, but imbalance alone is not necessarily a problem. The main problems are:

  * misleading metrics such as accuracy
  * insufficient positive examples
  * the model prioritizing the majority class
  * poorly chosen decision thresholds
  * probabilities becoming distorted after resampling


## Never balanced the test set

The validation and test sets should normally preserve the real class distribution

If fraud prevalence in production is 1%, the test set should also contaion approximately 1% fraud

Otherwise, metrics such as these may become unrealistic:

  - precision
  - accuracy
  - business cost
  - calibration
  - predicted workload

Resampling should normally be applied only to the training data


## Random oversampling

Oversampling increases the number of minority-class observations

Suppose the training set contains:

$$ 100~frauds, \quad 9900~legitimate~transactions $$


### Advetages

  * preserves all majority-class observations
  * gives the minority class more influence
  * simple to implement

### Disadvetages

  * duplicates existing information
  * may increase overfitiing
  * changes the apparent class prevalence
  * may damage probability calibration

After oversampling, 50% of training observations are fraudulent, but real prevalence is still only 1%


## Random undersampling

Undersampling removes majority-class observations

We might reduce:

$$ 9900~legitmate~transactions $$

to:

$$ 100~legitimate~transactions $$

The new training set would contain:

$$ 100~frauds, \quad 100~legitimate~transactions $$


### Adventages

  - reduces training time
  - may make the minority class easier to learn
  - useful when the majority class is extremely large

### Disadventages

  - discards potentially valunable information
  - may increase model variance
  - the retained negative examples may not represent the full population


## SMOTE

SMOTE creates synthetic minority observations by combinating neaby minority examples

Conceptally, if two minority obserations are:

$$ x_a, \quad x_b $$

a synthotic observation can be created between them:

$$ x_{new} = x_a + \lambda (x_b - x_a), \quad 0 \leq \lambda \leq 1 $$

SMOTE does not simply duplicate observations

However, in can create inrealisitc examples, especially when:

  - classes overlap
  - variables are categorical
  - data have temporial structure
  - minority observations contain noise


## Resampling leakage

Incorrect proces:

  1. apply SMOTE to the entire dataset
  2. perform cross-validation

Synthetic observations derived from one original transaction may appear in training while the original or a closely related synthetic transaction appears in validation

Correct process:

  1. create the cross-validation folds
  2. apply SMOTE only inside each training fold
  3. validate on the untouched validaton fold

In practice, this is usually implemented using a pipeline


## Class weights

Instead of changing the dataset, we can assing a larger loss to mistakes on the minority class

Weighted binary log loss is:

$$ L = -\frac{1}{n} \sum_{i=1}^{n} [w_1 y_i ln(p_i) + w_0(1 - y_i) ln(1 - p_i)] $$

Here:

  - $w_1$ is the positive-class weight
  - $w_0$ is the negative-class weight

A common balanced-weight formula is:

$$ w_c = \frac{n}{Kn_c} $$

where:

  - $n$ is the total number of obseravtions
  - $K$ is the number of classes
  - $n_c$ is the number of observations is class $c$

For 10,000 ibservations, 100 fraudsm and 9,90 legitimate transactions:

$$ w_{fraud} = \frac{10000}{2 \cdot 100} = 50, $$

$$ w_{legitimate} = \frac{10000}{2 \cdot 9900} \approx 0.5051 $$

The ratio is:

$$ \frac{50}{0.5051} \approx 99 $$

Thus, one fraudulent observation receives approximately 99 times as much weight as one legitimate observation


## Effects of class weights

Increasing the positive-class weight usually causes:

  - more positive predictions
  - higher recall
  - more false positives
  - lower spectificity
  - often lower precision

However, these effects are not guaranteed for every model and dataset

Class weights can also damage calibration because the model is trained using an artificial cost structure


## Class weights versus threshold selection

These methods affect differenct stages

### Class weights

Class weights change how the model is trained. They can change:

  - coefficients
  - ranking
  - decision boundary
  - predicted probabilities

### Threshold selection

Changing the threshold does not retrain the model. It only converts existing porbabilities or scores into classes

Sometimes am unweighted model with a carefully selected threshold works well. In other cases, class weights hepl the model learn useful minority-class patterns

Both approaches should be compared using validation data


### Recommend anti-fraud procedure

  1. Preserve the real prevalence in validation and test data
  2. Apply resampling only to training data
  3. Put resampling inside the cross-validation pipeline
  4. Compare an ordinary model, class weighting, and resampling
  5. Evaluate ROC-AUC and PR-AUC
  6. Check precision and recall at useful threshold
  7. Chech calibration after weighting or resampling
  8. Select the final approach using business cost

## Practice

### Task 1: Balanced class weights

A training set contains:

$$ n = 20000 $$

transactions:

$$ n_{fraud} = 200, \quad n_{legitimate} = 19800 $$

Calculate:

  1. fraud prevalence
  2. $w_{fraud}$
  3. $w_{legitimate}$
  4. the ratio $w_{fraud}/w_{legitimate}$
  5. explain why the minority class receives a larger weight


**Solution:**

1. fraud prevalence

$$ \frac{200}{20000} = 0.01 = 1\% $$

2. $w_{fraud}$

$$ w_{fraud} = \frac{20000}{200 \cdot 2} = 50 $$

3. $w_{legitimate}$

$$ w_{legitimate} = \frac{20000}{19800 \cdot 2} \approx 0.5051 $$

4. the ratio $w_{fraud}/w_{legitimate}$

$$ \frac{50}{0.5051} \approx 99 $$

5. because it is rarer class and more important, we need to make sure that model can catch observations like this


### Task 2: Oversampling

The original training set contains:

$$ 100~frauds, \quad 9900~legitimate~transactions $$

Fraud observations are oversampled until the training contain:

$$ 9900~fraud, \quad 9900~legitimate~transactions $$

Calculate:

  1. original fraud prevalence
  2. oversampled fraud prevalence
  3. whether the validation set should also be balanced
  4. whether $p = 0.50$ from the oversampled model necessatily means a real 50% fraud probability
  5. what should be checked before using these probabilities for cost-based deciisons


**Solution:**

1. original fraud prevalence

$$ \frac{100}{10000} = 0.01 = 1\% $$

2. oversampled fraud prevalence

$$ \frac{9900}{9900 + 9900} = 0.5 = 50\% $$

3. no, validation set shouldn't be balanced, because on this set we estimate how model can find real fraud transactions

4. no, it doesn't mean that the real fraud sare is 50%

5. model's classification ability and calibration should be checked before cost-based decision


### Task 3: Compare two models

The original test set contains 10,000 transactions and keeps the real prevalence

| Result | Model A | Model B |
| :--- | ---: | ---: |
| TP | 60 | 85 |
| FN | 40 | 15 |
| FP | 100 | 400 |
| TN | 9800 | 9500 |

For each model, calculate:

  1. accuracy
  2. precision
  3. recall
  4. FPR
  5. business cost when

  $$ C_{FN} = 500, \quad C_{FP} = 10 $$

Which model is preferable according to business metrics?


**Solution:**

1. accuracy

$$ accuracy_A = \frac{60 + 9800}{60 + 40 + 100 + 9800} = 0.986 $$

$$ accuracy_B = \frac{85 + 9500}{85 + 15 + 400 + 9500} = 0.9585 $$

2. precision

$$ precision_A = \frac{60}{60 + 100} = 0.375 $$

$$ precision_B = \frac{85}{85 + 400} \approx 0.175 $$

3. recall

$$ recall_A = \frac{60}{60 + 40}  = 0.6 $$

$$ recall_B = \frac{85}{85 + 15} = 0.85 $$

4. FPR

$$ FPR_A = \frac{100}{100 + 9800} \approx 0.01  $$

$$ FPR_B = \frac{400}{400 + 9500} \approx 0.04  $$

5. business costs

$$ costs_A = 500 \cdot 40 + 10 \cdot 100 = 21000 $$

$$ costs_B = 500 \cdot 15 + 10 \cdot 400 = 11500 $$


Model B is preferable because it has lower business costs, it happens because model B has higher recall score compared to model A


### Task 4: Identify correct precedures

For each situation, determine whether the procedure is correct and explain why

1. SMOTE is applied before dividing the data into cross-validation folds

  - this is incorrect, we corrupt real data, which should be used for test and validation

2. SMOTE is applied only to the training part of every fold

  - its okay, this allow to train model on different data, or stratified folds can be used

3. The validation and test sets are undersampled to produce 50% fraud prevalence

  - its incorrect, undersampling can be used only on train set, validation and test sets shouldn't be undersampled

4. Class weights are calculated using only the trainig data

  - its correct

5. Model calibration is checked on data with the real class prevalence

  - its correct

6. The classification threshold is selected using the final test set?

  - its incorrect, classification threshold selecting is performed on validation or calibration set


### Task 5: Conceptual questions

1. Why can oversampling damage probability calibration?

  - because calibration comaring real probability of positive observations in data and model predicted probability, oversampling distors data and that is a problem

2. What information can be lost through undersampling?

  - the main lost in undersampling strategy is real observations quantity and negative observations purity

3. How do class weights differ from threshold selection?

  - class weights is intended to point out at rare class in inbalanced data, threshold selection does not retrain the model. it only changes how existing scores are converted into predicted classes

4. Why should PR-AUC be examined when fraud is rare?

  - PR-AUC usually is a main metric in data with imbalanced classes because this metric focused on rare class

5. Does class imbalance automatically mean that the model cannot learn?

  - no, it doesn't, model still can work well

6. Can a model with lower accuracy have lower business cost?

  - yes, it can. It all depends on cost of FP and FN. This is particularly evident in anti-fraud, because data is classes are strong imbalanced. In these cases accuracy usually high because fraud is rare, but cost of missed fruad is too big and that the main reason why in cases like that we should sacrafice the accuracy